In [ ]:
#------------------------------------------------ Import Lib ----------------------------------------
import re
import os
import datetime
import requests
import pandas as pd
from bs4 import BeautifulSoup

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'KM CBCO' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd() ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


In [ ]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': []}


regdict={

        regulatorName+' 1': 'https://banque-comores.km/page/show/etablissements-de-credit',
        regulatorName+' 2': 'https://banque-comores.km/page/show/intermediaires-financiers',

        }


Typology={

       regulatorName + ' 1': 'Etablissements de credits',
       regulatorName + ' 2': 'Intermédiaires financiers',

        }


processdate = now.strftime('%Y-%m-%d')

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'}


#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    # print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    resp = requests.get(regdict[reg], headers=headers, verify=False, timeout=60)
    resp.raise_for_status()
    resp.encoding = 'utf-8'  # site serves UTF-8; keep accented French intact (no mojibake)
    soup = BeautifulSoup(resp.text, 'lxml')

    # main article body: excludes navigation, sidebar ("Sur la même rubrique") and footer
    content = soup.find('div', class_='blog__details-left')

    # page heading: 'Etablissements de crédit' / 'Intermédiaires financiers'
    heading = content.find('h3')

    # entities are the bullet points (<ul> siblings) following the heading;
    # the h4/<p> blocks below are descriptive text, not the list itself
    for ul in heading.find_next_siblings('ul'):
        for li in ul.find_all('li'):
            name_ = re.sub(r'\s+', ' ', li.get_text(' ', strip=True)).strip()
            name_ = name_.rstrip('.').strip()  # e.g. '... (MCTV-SA).' -> '... (MCTV-SA)'
            if not name_:
                continue
            sqldict['Name'].append(name_)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['RegulationType'].append('Regulated')

    sqldict = bourange_same_length_array(sqldict)


In [ ]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)

df = df[df['Name']!='']

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(tempfolder, filename)))
